In [ ]:


import requests
import pandas as pd
import re
import time
import logging
from typing import List, Dict, Optional, Any
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# تنظیمات نمایش لاگ‌ها در کنسول
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

class BamaVehicleScraper:
    """کلاسی برای استخراج داده‌های خودرو از وب‌سایت باما با استفاده از API پنهان"""

    TARGET_API = "https://bama.ir/cad/api/search"

    def __init__(self, vehicle_query: str, min_year: int, target_count: int):
        self.vehicle_query = vehicle_query
        self.min_year = min_year
        self.target_count = target_count
        self._client = self._init_http_client()
        self.extracted_dataset: List[Dict[str, Any]] = []

    def _init_http_client(self) -> requests.Session:
        """ساخت و تنظیم یک سشن شبکه با قابلیت تلاش مجدد (Retry) در صورت قطعی"""
        client = requests.Session()
        retries = Retry(total=3, backoff_factor=1, status_forcelist=[500, 502, 503, 504])
        client.mount('https://', HTTPAdapter(max_retries=retries))

        client.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
            'Accept': 'application/json, text/plain, */*',
            'Referer': 'https://bama.ir/',
            'Cookie': 'x-device-target=desktop; evonex-locale=fa'
        })
        return client

    @staticmethod
    def _sanitize_string(raw_str: Optional[str]) -> Optional[str]:
        """پاک‌سازی متن از کاراکترهای اضافی و فاصله‌های نامرئی"""
        return raw_str.strip().replace('\u200c', ' ').replace('\xa0', ' ') if raw_str else None

    @staticmethod
    def _parse_numeric(raw_val: Any) -> Optional[int]:
        """استخراج اعداد از داخل رشته‌های متنی (مثل قیمت و کارکرد)"""
        if not raw_val:
            return None
        digits_only = re.sub(r'\D', '', str(raw_val))
        return int(digits_only) if digits_only else None

    def _is_valid_model(self, year_str: Optional[str]) -> bool:
        """بررسی اینکه آیا سال ساخت خودرو از حداقل سال مد نظر بیشتر است یا خیر"""
        if not year_str:
            return False
        try:
            return int(year_str) > self.min_year
        except ValueError:
            return False

    def _parse_listing(self, item: dict) -> Optional[Dict[str, Any]]:
        """پردازش و استخراج فیلدهای مورد نیاز از ساختار JSON یک آگهی"""
        details = item.get('detail', {})
        price_data = item.get('price', {})

        if not self._is_valid_model(details.get('year')):
            return None

        return {
            'Description': self._sanitize_string(details.get('title')),
            'Production Year': details.get('year'),
            'Transmission': self._sanitize_string(details.get('transmission')),
            'Mileage': self._parse_numeric(details.get('mileage')),
            'Color': self._sanitize_string(details.get('color')),
            'Price': self._parse_numeric(price_data.get('price'))
        }

    def execute(self) -> pd.DataFrame:
        """تابع اصلی برای اجرای عملیات اسکریپینگ تا رسیدن به تعداد تارگت"""
        current_page = 1

        while len(self.extracted_dataset) < self.target_count:
            logging.info(f"درحال بررسی صفحه {current_page} برای خودروی {self.vehicle_query}...")

            params = {
                'vehicle': self.vehicle_query,
                'pageIndex': current_page
            }

            try:
                response = self._client.get(self.TARGET_API, params=params, timeout=10)
                response.raise_for_status()
                payload = response.json()
            except requests.RequestException as err:
                logging.error(f"خطای شبکه در دریافت صفحه {current_page}: {err}")
                break

            ads_list = payload.get('data', {}).get('ads', [])
            if not ads_list:
                logging.warning("آگهی جدیدی یافت نشد یا به انتهای صفحات رسیدیم.")
                break

            for ad in ads_list:
                if len(self.extracted_dataset) >= self.target_count:
                    break

                parsed_car = self._parse_listing(ad)
                if parsed_car:
                    self.extracted_dataset.append(parsed_car)

            current_page += 1
            time.sleep(1.5)  

        logging.info(f"عملیات موفق! تعداد {len(self.extracted_dataset)} رکورد استخراج شد.")
        return pd.DataFrame(self.extracted_dataset)


# ---------------------------------------------------------
# Execution Block
# ---------------------------------------------------------
if __name__ == "__main__":
    # استخراج ۵۰ رکورد خودروی سمند مدل بالای ۱۳۸۵
    samand_scraper = BamaVehicleScraper(
        vehicle_query="samand",
        min_year=1385,
        target_count=50
    )
    
    df_result = samand_scraper.execute()
    
    if not df_result.empty:
        export_filename = 'Bonus_ADS_Samand_Bama.xlsx'
        df_result.to_excel(export_filename, index=False)
        logging.info(f"✅ فایل نهایی با نام '{export_filename}' ذخیره شد.")
        display(df_result.head())
    else:
        logging.warning("⚠️ داده‌ای برای ذخیره یافت نشد.")

INFO: درحال بررسی صفحه 1 برای خودروی samand...
INFO: درحال بررسی صفحه 2 برای خودروی samand...
INFO: عملیات موفق! تعداد 50 رکورد استخراج شد.
INFO: ✅ فایل نهایی با نام 'Bonus_ADS_Samand_Bama.xlsx' ذخیره شد.


,Description,Production Year,Transmission,Mileage,Color,Price
0,سمند، سورن,1403,دنده ای,25000.0,سفید / داخل مشکی,1750000000
1,سمند، سورن,1403,دنده ای,43000.0,سفید / داخل مشکی,1770000000
2,سمند، سورن,1404,دنده ای,3000.0,سفید / داخل مشکی,1700000000
3,سمند، LX,1386,دنده ای,400000.0,نقره ای / داخل کرم,600000000
4,سمند، LX,1387,دنده ای,388089.0,نقره ای / داخل قهوه ای,430000000
